In [ ]:
import time

start_time = time.time()
print(f"Start time recorded: {start_time}")

Start time recorded: 1765867807.266835


<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/04_particle_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [ ]:
do = False # @param{type:"boolean"}
if do:
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

In [ ]:
if do:
    !git clone https://github.com/phonchi/CryoParticleSegment.git

    !wget -O /content/CryoParticleSegment/Modeling/convcrf.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
    !wget -O /content/CryoParticleSegment/Modeling/dataset.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/dataset.py
    !wget -O /content/CryoParticleSegment/Modeling/model.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/model.py
    !wget -O /content/CryoParticleSegment/Modeling/trainer.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/trainer.py

In [ ]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
if do:
    #%git clone https://github.com/netw0rkf10w/CRF.git
    %cd CryoParticleSegment/Modeling/CRF_main
    !python setup.py clean --all
    !rm -rf build/
    !python setup.py build_ext --inplace --force
    !python setup.py install

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [ ]:
if do:
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [ ]:
# @title  { display-mode: "form" }

INPUT_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
IMAGE_DIR = INPUT_IMAGE_DIR
# @markdown ---

use_denoised_as_pariwise = False # @param {type : "boolean"}
dnzd_pw = use_denoised_as_pariwise
DENOISED_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
# @markdown ---

LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_test/10017/unet_eb5_dice_CRF" # @param {type:"string"}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rm: cannot remove '/content/sample_data': No such file or directory


In [ ]:
IMAGE_DIR = "/content/image_dir"

In [ ]:
%cd /content/

/content


### ✅ Packages Handling

In [ ]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR

In [ ]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery
from dataset import reconstruct_patched, collate_fn
from model import create_model
from trainer import CryoEMEvaluator
from trainer import CryoEMTrainerWithScheduler, tqdm_plugin_for_Trainer

## ⭐ Main

### ✅ Setting

In [ ]:
# @markdown Parameters.

NUM_CLASSES = 2
EPOCHS = 50
BATCH = 8
CROP_SIZE = (512, 512)
LR = 1e-3

RLR_PATIENCE = 3
ES_PATIENCE = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

## ⭐ Convcrf wtih FCN finetuned on cryoem

### ✅ Model

## The model

In [ ]:
# @title  { display-mode: "form" }

architecture = "Unet++" # @param {type:"string"}
encoder = "timm-efficientnet-b5" # @param {type:"string"}
pretrained = True # @param {type:"boolean"}
solver = "fw" # @param {type:"string"}
use_unary_only = True # @param {type:"boolean"}


In [ ]:
import segmentation_models_pytorch as smp

if pretrained:
  weights = "imagenet"
else:
  weights = None

if architecture == "Unet++":
    backbone = smp.UnetPlusPlus(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` or `advprop` for pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )

elif architecture == "Deeplab":
    backbone = smp.DeepLabV3(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )
else:
    print("Architecture not supported")
    raise NotImplementedError

model = create_model(backbone, addout=True) #crf_args

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

In [ ]:
import CRF
import torch.nn as nn
from model import setup_crf, create_fwcrf_model

# Example usage
solver = 'fw'  # Assuming the solver type is defined

crf = setup_crf(solver, NUM_CLASSES)
model_post = create_fwcrf_model(model.backbone, crf, use_unary_only=True)

CRF solver: fw
x0_weight: 0.0
FrankWolfeParams: 
	 scheme:	 fixed 
	 stepsize:	 1.0 (for the 'fixed' scheme) 
	 regularizer:	 l2
	 lambda_:	 1.0
	 lambda_learnable:	 False
	 x0_weight:	 0.5
	 x0_weight_learnable:	 False
Non-trainable lambda for Frank-Wolfe: 1.0
Non-trainable x0_weight for Frank-Wolfe: 0.5
Potts: remove random weights.
Add 1.0 to spatial_weight diagonal
Add 1.0 to bilateral_weight diagonal
Add -1.0 to compatibility diagonal


## ⭐ Evaluate

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
from dataset import reconstruct_patched

def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy

In [ ]:
from torch.utils.data import ConcatDataset

train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
if dnzd_pw == False:
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE)
else:
    dnzd_train_dir = os.path.join(DENOISED_IMAGE_DIR, 'train')
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_train_dir, filenames=train_filenames, crop_size=CROP_SIZE)
train_loader = DataLoader(train_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
if dnzd_pw == False:
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE)
else:
    dnzd_val_dir = os.path.join(DENOISED_IMAGE_DIR, 'val')
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_val_dir, filenames=val_filenames, crop_size=CROP_SIZE)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)

test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
if dnzd_pw == False:
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, filenames=test_filenames, crop_size=CROP_SIZE)
else:
    dnzd_test_dir = os.path.join(DENOISED_IMAGE_DIR, 'test')
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, denoised_dir = dnzd_test_dir, filenames=test_filenames, crop_size=CROP_SIZE)

test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


full_filenames = np.concatenate((train_filenames, val_filenames, test_filenames))
full_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, pin_memory=True, collate_fn=collate_fn)

In [ ]:
shape = None
for i1, i2, i3, i4, i5 in val_loader: #test loader and reconstruct
    shape = i5.shape
    break

In [ ]:
from PIL import Image

def normalize(im):
    max_mrc=np.max(im)
    min_mrc=np.min(im)
    img_original=(255*((im-min_mrc)/(max_mrc-min_mrc))).astype(np.uint8)
    return(img_original)

def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

In [ ]:
from tqdm import tqdm

fnames = []
n = len(full_dataset)
processed_micrographs = np.empty((n, shape[1], shape[2]), dtype=np.float32)

# Use tqdm to wrap your loop for a progress bar
for idx, (test_image, _, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing images"):
    name = full_filenames[idx][:-4]
    fnames.append(name)
    # Determine the directory and load the micrograph
    if os.path.exists(f"{IMAGE_DIR}/test/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/test/{name}.npy"
    elif os.path.exists(f"{IMAGE_DIR}/train/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/train/{name}.npy"
    else:
        micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
    micrograph = np.load(micrograph_path)
    processed_micrograph = preprocess_and_crop(micrograph)
    # Place the processed micrograph directly into the preallocated array
    processed_micrographs[idx] = processed_micrograph

Processing images: 100%|██████████| 84/84 [00:27<00:00,  3.07it/s]


In [ ]:
processed_micrographs.shape

(84, 3840, 3840)

In [ ]:
np.save(f"{RESULT_DIR}/processed_micrographs.npy", processed_micrographs)

In [ ]:
processed_micrographs = np.load(f"{RESULT_DIR}/processed_micrographs.npy")

In [ ]:
del model
torch.cuda.empty_cache()

In [ ]:
model = model_post

In [ ]:
RESULT_DIR

'/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_test/10017/unet_eb5_dice_CRF'

In [ ]:
import torch.nn.functional as F

label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)

checkpoint_paths = [path for path in os.listdir(RESULT_DIR) if '.pt' in path]
checkpoint_path = checkpoint_paths[-1]
state_dict_path = f"{RESULT_DIR}/{checkpoint_path}"
state_dict = torch.load(state_dict_path, map_location=torch.device(device))
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

mini_batch_size = 18  # Number of patches to process at once
n = len(full_dataset)
label_images = np.empty((n, shape[1], shape[2]), dtype=np.uint8)  # Preallocated array

# Iterate through the dataset with tqdm for progress tracking
for idx, (test_image, _, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing dataset"):
    # Process in batches
    with torch.no_grad():
        inputs = test_image.to(device)
        num_batches = (inputs.size(0) + mini_batch_size - 1) // mini_batch_size
        patched_outputs = []

        for batch_idx in range(num_batches):
            start_idx = batch_idx * mini_batch_size
            end_idx = min(start_idx + mini_batch_size, inputs.size(0))
            patch_input = inputs[start_idx:end_idx].to(device)
            output = model(patch_input)['out']
            patched_outputs.append(output.cpu())  # Minimize device memory usage

            # Cleanup as soon as not needed
            del patch_input, output
            torch.cuda.empty_cache()

        outputs = torch.cat(patched_outputs)  # Combine outputs
        probabilities = F.softmax(outputs, dim=1)
        class1_probabilities = probabilities[:, 1, :, :]  # Assuming class 1 is the target
        pred_image = reconstruct_patched(class1_probabilities.unsqueeze(1), grid).float()

        output_image = normalize(pred_image.squeeze().numpy())

        # Cleanup large temporary variables
        del patched_outputs, outputs, probabilities, class1_probabilities, pred_image
        torch.cuda.empty_cache()

    # Store the output image directly in the preallocated array
    label_images[idx] = output_image

    if idx % 30 == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(output_image, cmap='inferno', alpha=0.4)
        plt.show()
    del output_image
    torch.cuda.empty_cache()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
np.save(f"{RESULT_DIR}/label_images.npy", label_images)

In [ ]:
!cp {RESULT_DIR}/label_images.npy .

In [ ]:
label_images = np.load(f"{RESULT_DIR}/label_images.npy")

In [ ]:
label_images.shape

(84, 3840, 3840)

In [ ]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}

In [ ]:
!cp {RESULT_DIR}/best_config.txt .

In [ ]:
with open("best_config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split(": ", 1)
        if key == "cv_config":
            cv_config = eval(value)
        elif key == "tp_config":
            tp_config = eval(value)
        elif key == "nms_config":
            nms_config = eval(value)

### Actual seg

In [ ]:
radius = RADIUS

In [ ]:
from skimage import filters, segmentation, morphology
import skimage as ski
from skimage import img_as_float
from center_finding import normalize, min_rect_circle, eliminate_near
import cv2

In [ ]:
from skimage import img_as_float
from skimage.filters import threshold_local

cv_list = []
for img in label_images:
    output_image = img_as_float(img)
    block_size = int(round(radius * 1.6)) | 1  # Ensure it's an odd number
    local_thresh = filters.threshold_local(output_image, block_size, method='gaussian', offset=0)
    image_seg = output_image > local_thresh
    thresh1 = ski.util.img_as_ubyte(image_seg)

    contr_min = radius*cv_config[1]
    kernel_size = int(radius / cv_config[0])  # Example ratio
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    thresh1 = cv2.erode(thresh1, kernel, iterations=1)

    contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_array = [c for c in contours]

    # Filter out small/large region and find bounding box with center
    c_ = np.array([cv2.contourArea(contour) for contour in contours])
    aa = (c_>contr_min) & (c_<500000)
    aa = aa.tolist()
    c_full_list = [d for d, keep in zip(cont_array, aa) if keep]
    c_list = (list(map(lambda x: min_rect_circle(x, radius), c_full_list)))
    c_list = [x for x in c_list if x is not None]
    cv_list.append(c_list)
    print(len(c_full_list), len(c_list))

530 454
512 435
489 386
393 372
413 376
508 437
393 382
454 409
519 445
561 488
510 474
515 459
522 453
477 441
562 480
530 473
505 425
552 515
516 482
582 521
552 469
529 448
544 506
565 489
464 405
477 377
483 381
522 432
485 385
488 466
510 400
494 390
520 435
531 442
494 398
529 445
490 409
535 462
545 516
526 449
491 461
534 432
548 472
390 358
378 362
387 348
399 376
576 517
572 509
505 424
545 484
542 468
541 471
549 490
537 477
469 406
510 438
547 453
480 389
448 347
463 442
461 420
459 411
444 402
433 395
436 414
441 401
447 398
397 349
417 355
438 388
407 392
442 388
444 401
422 390
495 438
443 407
421 368
432 395
448 420
419 354
445 378
418 353
455 396


In [ ]:
import matplotlib
for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    #if idx == 6:
    #    break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in cv_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
!pip install starfile -qq

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/CRF-0.0.1-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [ ]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in cv_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [ ]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,1509,3948,00000@Falcon_2012_06_12-14_33_35_0.mrc,Falcon_2012_06_12-14_33_35_0.mrc
1,1235,3948,00001@Falcon_2012_06_12-14_33_35_0.mrc,Falcon_2012_06_12-14_33_35_0.mrc
2,227,3948,00002@Falcon_2012_06_12-14_33_35_0.mrc,Falcon_2012_06_12-14_33_35_0.mrc
3,157,3944,00003@Falcon_2012_06_12-14_33_35_0.mrc,Falcon_2012_06_12-14_33_35_0.mrc
4,427,3944,00004@Falcon_2012_06_12-14_33_35_0.mrc,Falcon_2012_06_12-14_33_35_0.mrc
...,...,...,...,...
2455,2226,184,00432@Falcon_2012_06_13-03_22_02_0.mrc,Falcon_2012_06_13-03_22_02_0.mrc
2456,3824,153,00433@Falcon_2012_06_13-03_22_02_0.mrc,Falcon_2012_06_13-03_22_02_0.mrc
2457,1489,200,00434@Falcon_2012_06_13-03_22_02_0.mrc,Falcon_2012_06_13-03_22_02_0.mrc
2458,405,151,00435@Falcon_2012_06_13-03_22_02_0.mrc,Falcon_2012_06_13-03_22_02_0.mrc


In [ ]:
import starfile
starfile.write(df, f'{RESULT_DIR}/cv_particles.star')

### TrackPy

In [ ]:
cv_list

[[(1381, 3820),
  (1107, 3820),
  (99, 3820),
  (29, 3816),
  (299, 3816),
  (2414, 3817),
  (625, 3815),
  (3587, 3811),
  (3038, 3813),
  (1186, 3801),
  (2826, 3783),
  (2697, 3788),
  (1562, 3781),
  (3691, 3768),
  (1701, 3767),
  (3822, 3726),
  (1295, 3731),
  (976, 3734),
  (26, 3691),
  (3425, 3685),
  (471, 3687),
  (1791, 3682),
  (367, 3668),
  (1134, 3650),
  (2769, 3659),
  (1488, 3657),
  (3049, 3646),
  (1019, 3628),
  (1002, 3563),
  (756, 3564),
  (1127, 3539),
  (3231, 3549),
  (42, 3529),
  (678, 3516),
  (2369, 3490),
  (3001, 3496),
  (3476, 3492),
  (137, 3485),
  (1581, 3475),
  (2820, 3452),
  (2766, 3464),
  (602, 3470),
  (3799, 3493),
  (1776, 3459),
  (764, 3429),
  (1477, 3437),
  (2264, 3415),
  (1000, 3414),
  (16, 3408),
  (2932, 3404),
  (2640, 3411),
  (2527, 3442),
  (1921, 3401),
  (516, 3408),
  (2805, 3360),
  (889, 3361),
  (671, 3347),
  (211, 3360),
  (3048, 3349),
  (1606, 3341),
  (1980, 3316),
  (2140, 3330),
  (3373, 3317),
  (333, 3313),
 

In [ ]:
tp_config

(0.25, 0.4)

In [ ]:
import trackpy as tp

In [ ]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = round(TrackParticleSize*tp_config[1])
scale = tp_config[0]  # Scale factor (0.5 means reducing the size to half)

In [ ]:
tp_list = []
for img in label_images:
    output_image = img
    small_image = cv2.resize(output_image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    # Adjust parameters based on the scale
    small_sep = int(sep * scale)
    small_diameter = int(TrackParticleSize * scale)
    if small_diameter % 2 == 0:
        small_diameter += 1
    coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)
    #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]
    coorTrack['x'] *= (1/scale)
    coorTrack['y'] *= (1/scale)
    coords = coorTrack[['x', 'y']].values
    tp_list.append(coords)
    print(len(coords))

604
559
611
388
424
570
371
465
590
617
505
523
580
485
639
566
554
516
503
592
600
627
562
633
517
598
609
623
607
478
617
641
585
595
612
614
556
605
519
595
441
626
600
387
365
403
387
594
627
594
587
569
570
577
575
525
583
674
641
605
439
474
462
440
434
426
450
483
439
473
466
384
487
451
432
516
438
480
416
436
490
498
475
502


In [ ]:
tp_list

[array([[1421.67740368,  107.50697177],
        [3215.38717137,   49.99528524],
        [1013.24611264,   64.27163158],
        ...,
        [2702.21665644, 3776.89336363],
        [3205.72605893, 3742.3842081 ],
        [3689.88653291, 3764.17077063]]),
 array([[ 223.12018756,   79.84491678],
        [1065.78981507,   78.55603091],
        [2400.02804557,  102.10962606],
        ...,
        [3600.35311971, 3785.29976002],
        [ 477.17741905, 3764.8222419 ],
        [2778.21340803, 3782.51558954]]),
 array([[ 281.54928018,   61.5111533 ],
        [1053.53595647,   56.46298068],
        [ 921.89000866,   77.78771855],
        ...,
        [3551.00073826, 3793.4697529 ],
        [1969.37028247, 3768.18215235],
        [2746.44124045, 3780.5208185 ]]),
 array([[2922.06368574,   61.40555018],
        [ 874.71729162,   51.70062138],
        [2249.55652912,   51.62003736],
        [3564.23607683,   59.90649927],
        [1260.35036149,   64.38390746],
        [1422.01943858,   88.180256

In [ ]:
import matplotlib
for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in tp_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in tp_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [ ]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,1549.677404,235.506972,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,3343.387171,177.995285,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,1141.246113,192.271632,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,3608.544392,190.313051,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,1467.740739,222.790432,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
44371,188.168074,3824.796704,00497@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
44372,766.343783,3831.903860,00498@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
44373,1521.828111,3909.539132,00499@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
44374,3389.397998,3907.574765,00500@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [ ]:
import starfile
starfile.write(df, f'{RESULT_DIR}/tp_particles.star')

### Nonmax

In [ ]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [ ]:
nms_config

(0.5, 0.6)

In [ ]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more
pCan=nms_config[0] #the grid size smaller will generate more candidate
overlap=nms_config[1] #allow for overlap
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [ ]:
nms_list = []
for idx, img in enumerate(label_images):
    # if idx == 6:
    #     break
    heatArr=pad_image(img)
    heatArr=reNorm(heatArr, nSep)
    heatArr=reLev(heatArr,level)
    gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

    # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
    gsize=psize*pCan
    heat=heatArr.astype(np.float32)
    gaus=gaus.astype(np.float32)
    score=np.zeros(heat.shape).astype(np.float32)
    [sizex,sizey]=heat.shape
    sizex = int(sizex)
    sizey = int(sizey)
    psize = int(psize)
    gsize = int(gsize)

    func = scoreGpu

    tx=16
    ty=16
    bx=(sizex-1)//tx+1
    by=(sizey-1)//ty+1
    print('get score gpu',tx, ty, bx, by, gsize, psize)


    func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
        block=(tx, ty, 1), grid=(int(bx), int(by)))

    # # Calculate necessary dimensions and convert all to int
    numx = int((sizex - 1) // gsize + 1)
    numy = int((sizey - 1) // gsize + 1)
    num = numx * numy
    tnum = num * 5

    # # Create the result array
    res = np.zeros(tnum).astype(np.float32)

    # # Block and grid dimensions
    bx = (numx - 1) // tx + 1
    by = (numy - 1) // ty + 1

    # Get the function from the module
    func = getMax

    # Call the function, ensure all parameters are the correct type
    func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
        block=(16, 16, 1), grid=(bx, by, 1))

    # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
    niter = np.int32(nIter)
    gsize = np.int32(gsize)
    sizex = np.int32(sizex)
    sizey = np.int32(sizey)
    numx = np.int32(numx)
    tnum = np.int32(num)

    print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
    func = getMax3
    # Ensuring the correct parameter order and type for the kernel invocation
    func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
            block=(16, 16, 1), grid=(bx, by, 1))

    canList=reshape(res,num)

    print('Number of Particles before:', len(canList))
    if(len(canList)>pnum):
        canList=canList[:pnum]


    canList=cleanCanList(canList,overlap,psize)
    #canList=reCan(canList,ratio)
    print('Number of Particles:', len(canList))
    nms_list.append([(r[1],r[0]) for r in canList])

get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3468
Number of Particles: 651
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3463
Number of Particles: 604
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3471
Number of Particles: 659
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3268
Number of Particles: 457
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3283
Number of Particles: 478
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3423
Number of Particles: 637
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3

In [ ]:
import matplotlib
for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
    # else:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in nms_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in nms_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [ ]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,1140.8,1784.8,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,2800.0,3098.2,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,2444.6,3782.4,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,2300.0,3760.0,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,340.6,2344.6,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
48736,2112.0,1087.0,00569@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
48737,3776.0,3008.0,00570@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
48738,3695.6,2940.0,00571@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
48739,470.0,1589.0,00572@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [ ]:
starfile.write(df, f'{RESULT_DIR}/nms_particles.star')

## Watershed

In [ ]:
# @title particle finding function of watershed
import cv2
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops

""" EXAMPLE OF ndimage.distance_transform_edt()
a = np.array(([0,1,1,1,1],
              [0,0,1,1,1],
              [0,1,1,1,1],
              [0,1,1,1,0],
              [0,1,1,0,0]))
ndimage.distance_transform_edt(a)
array([[ 0.    ,  1.    ,  1.4142,  2.2361,  3.    ],
       [ 0.    ,  0.    ,  1.    ,  2.    ,  2.    ],
       [ 0.    ,  1.    ,  1.4142,  1.4142,  1.    ],
       [ 0.    ,  1.    ,  1.4142,  1.    ,  0.    ],
       [ 0.    ,  1.    ,  1.    ,  0.    ,  0.    ]])
"""

def calculate_shape_metrics(region):
    """Computes Sphericity, Circularity, and Inverse Slenderness."""
    area = region.area
    perimeter = region.perimeter
    if perimeter == 0: return 0, 0, 0

    circularity = (4 * np.pi * area) / (perimeter ** 2)

    d_eq = np.sqrt(4 * area / np.pi)
    d_cir = region.major_axis_length
    sphericity = d_eq / d_cir if d_cir > 0 else 0

    major = region.major_axis_length
    minor = region.minor_axis_length
    inv_slenderness = minor / major if major > 0 else 0

    return sphericity, circularity, inv_slenderness


def detect_particles_dt(prob_map_input, particle_radius=28,
                             peak_threshold_ratio=0.6, min_dist_ratio=0.4): # changed: removed min_solidity
    """
    DT Watershed with Adaptive Shape Thresholding.

    min_dist_ratio: Min distance between 2 peaks (fraction of radius). e.g., 0.4 * 28 = 11 pixels.
    peak_threshold_ratio: multiply * with the particle_radius, and if the pixel is larger than that, it will be allow to be a candidate of the peak.

    """
    # 1. Robust Normalization
    prob_map = prob_map_input.astype(np.float32)
    height, width = prob_map.shape[:2]
    if prob_map.max() > 1.0: prob_map /= 255.0

    binary_mask = (prob_map > 0.5).astype(np.uint8)
    if np.sum(binary_mask) == 0: binary_mask = (prob_map > 0.05).astype(np.uint8)
    if np.sum(binary_mask) == 0: return []

    # 2. DT transform with scipy ndimage
    distance = ndi.distance_transform_edt(binary_mask)

    # 3. Find Markers

    # Separation Distance
    min_dist = int(particle_radius * min_dist_ratio) # multiply min_dist_ratio with radius to set a min distance must be followed

    # Height Threshold (Bundled with Radius)
    # e.g., If radius=28 and ratio=0.5, we require the peak be at least of 14 pixels.
    min_peak_height = particle_radius * peak_threshold_ratio

    local_max_coords = peak_local_max(
        distance,                    # Pass the DT map
        min_distance=min_dist,
        labels=binary_mask,
        threshold_abs=min_peak_height # Threshold in Pixels
    )

    markers = np.zeros(distance.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords):
        markers[r, c] = i + 1 # add markers

    # 4. Watershed
    # pass the (-) raw distance.
    labels = watershed(-distance, markers, mask=binary_mask)

    # 5. Filter & Extract (Adaptive Phase)
    expected_area = np.pi * (particle_radius ** 2)
    min_area = expected_area * 0.2
    max_area = expected_area * 2.8
    boundary_margin = particle_radius * 1.2 # prevent from picking particles at the margin

    regions = regionprops(labels)
    candidates = []
    metrics_sp = []
    metrics_cr = []
    metrics_isd = []

    for region in regions:
        # Basic Pre-filtering (Area only, Solidity removed) # changed
        is_valid_area = (min_area <= region.area <= max_area)

        if is_valid_area:
            sp, cr, isd = calculate_shape_metrics(region)
            candidates.append({'region': region, 'sp': sp, 'cr': cr, 'isd': isd})
            metrics_sp.append(sp)
            metrics_cr.append(cr)
            metrics_isd.append(isd)

    if not candidates:
        return []

    # 6. Calculate Dynamic Thresholds
    def get_adaptive_range(metric_list):
            """
            Calculates both Lower and Upper adaptive thresholds.

            Lower Bound (For 'Higher is Better' quality):
            - Uses Top 50% Mean (Good particles).
            - Stricter = Higher Value (np.max).

            Upper Bound (For outlier/anomaly detection):
            - Uses Lower 50% Mean (Background/Noise reference).
            - Stricter = Lower Value (np.min).
            """
            if not metric_list: return 0.0, float('inf')

            # 1. Calculate Statistics
            max_val = np.max(metric_list)
            min_val = np.min(metric_list)
            data_range = max_val - min_val

            # Sort for quantiles
            sorted_vals = sorted(metric_list)
            n = len(sorted_vals)
            mid = max(1, n // 2)

            # ---------------------------------------------------------
            # A. Lower Threshold (Min Acceptable Quality)
            # ---------------------------------------------------------
            # Use Top 50% (The "Good" half)
            top_half = sorted_vals[mid:]
            mean_top50 = np.mean(top_half)

            # Logic: Mean - (Range * 0.5)
            calc_lower = mean_top50 - (data_range * 0.8)
            # Safety: Don't go below 80% of the good mean
            floor_lower = mean_top50 * 0.7

            # Take the STRICTER (Higher) bound
            thresh_min = np.max([calc_lower, floor_lower])

            # ---------------------------------------------------------
            # B. Upper Threshold (Max Acceptable / Outlier Cutoff)
            # ---------------------------------------------------------
            # Use Lower 50% (The "Bad" or "Small" half) - As requested
            lower_half = sorted_vals[:mid]
            mean_lower50 = np.mean(lower_half)

            # Logic: Mean + (Range * 0.5)
            # "instead of minus add it back"
            calc_upper = mean_lower50 + (data_range * 0.8) # perhaps 0.7 ~ 0.85 will be better (need more trial)

            # Safety: Don't go above 120% of the lower mean
            # "min with 1.2mean of lower 50%"
            ceiling_upper = mean_lower50 * 1.4

            # Take the Looser (Lower) bound
            thresh_max = np.max([calc_upper, ceiling_upper])

            # Sanity Check: Ensure min <= max. If crossed, rely on min.
            if thresh_max < thresh_min:
                thresh_max = float('inf')

            return thresh_min, thresh_max

    # The adaptive cutoff values for Sphericity, Circularity, and Inv-Slenderness.
    min_sp, max_sp = get_adaptive_range(metrics_sp)
    min_cr, max_cr = get_adaptive_range(metrics_cr)
    min_isd, max_isd = get_adaptive_range(metrics_isd)


    # 7. Final Selection
    valid_particles = []
    for cand in candidates:
        # Must pass ALL adaptive shape thresholds
        if (min_sp <= cand['sp'] <= max_sp and
            min_cr <= cand['cr'] <= max_cr and
            min_isd <= cand['isd'] <= max_isd):

            region = cand['region']
            cy, cx = region.centroid

            if (cx < boundary_margin or cx > (width - boundary_margin) or
                cy < boundary_margin or cy > (height - boundary_margin)):
                continue

            score = prob_map[int(cy), int(cx)]
            valid_particles.append([int(cx), int(cy), float(score)])

    return valid_particles

In [ ]:
import json
import os

# ==========================================
# 1. SETUP: Load Best Parameters from JSON
# ==========================================
param_file_path = os.path.join(RESULT_DIR, 'best_watershed_params.json')

if not os.path.exists(param_file_path):
    raise FileNotFoundError(f"❌ Parameter file not found at: {param_file_path}\nPlease run Notebook 03 to generate it.")

with open(param_file_path, 'r') as f:
    best_params = json.load(f)

print(f"📂 Loaded params from: {param_file_path}")
print(f"📄 Content: {best_params}")

RADIUS = 28 # I have to adust with the above given  RADUIS of the notebook

# 2. Get Ratios
if 'config' in best_params and isinstance(best_params['config'], (list, tuple)):
    BEST_PEAK_RATIO = best_params['config'][0]
    BEST_DIST_RATIO = best_params['config'][1]
else:
    BEST_PEAK_RATIO = best_params.get('thresh')
    BEST_DIST_RATIO = best_params.get('dist')

print(f"\n🚀 Running Inference with: Radius={RADIUS}, Peak={BEST_PEAK_RATIO}, Dist={BEST_DIST_RATIO}")

# ==========================================
# 2. INFERENCE LOOP (Sequential)
# ==========================================
final_predictions = []

for idx, img in enumerate(label_images):
    try:
        # Run detection using the loaded parameters
        preds = detect_particles_dt(
            img,
            particle_radius=int(RADIUS),
            peak_threshold_ratio=float(BEST_PEAK_RATIO),
            min_dist_ratio=float(BEST_DIST_RATIO)
        )
        final_predictions.append(preds)

        print(f"Processed image {idx}/{len(label_images)} - Found {len(preds)} particles")

    except Exception as e:
        print(f"❌ Error on image {idx}: {e}")
        final_predictions.append([])

print(f"\n✅ Inference Complete! Extracted particles for {len(final_predictions)} images.")

📂 Loaded params from: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_test/10017/unet_eb5_dice_CRF/best_watershed_params.json
📄 Content: {'method': 'dt_only_no_ar', 'thresh': 0.4, 'dist': 0.8, 'f_score': 0.9612428997790701, 'map': 0.7787435054779053}

🚀 Running Inference with: Radius=28, Peak=0.4, Dist=0.8
Processed image 0/84 - Found 643 particles
Processed image 1/84 - Found 597 particles
Processed image 2/84 - Found 637 particles
Processed image 3/84 - Found 428 particles
Processed image 4/84 - Found 440 particles
Processed image 5/84 - Found 595 particles
Processed image 6/84 - Found 378 particles
Processed image 7/84 - Found 478 particles
Processed image 8/84 - Found 593 particles
Processed image 9/84 - Found 639 particles
Processed image 10/84 - Found 519 particles
Processed image 11/84 - Found 506 particles
Processed image 12/84 - Found 623 particles
Processed image 13/84 - Found 503 particles
Processed image 14/84

In [ ]:
import matplotlib
for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
    # else:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in final_predictions[idx]:
            x, y, _ = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
"""
# Visualize the first 3 results
for idx in range(3):
    if idx >= len(final_predictions): break

    coords = final_predictions[idx]
    bg_image = target_images[idx] # Or load the corresponding micrograph if available

    plt.figure(figsize=(10, 10))
    plt.imshow(bg_image, cmap='gray')

    if coords:
        coords_np = np.array(coords)
        # coords are [x, y, score]
        plt.scatter(coords_np[:, 0], coords_np[:, 1], c='red', s=20, marker='x', label='Prediction')

    plt.title(f"Test Image {idx} | Count: {len(coords)}")
    plt.axis('off')
    plt.legend()
    plt.show()
    """

'\n# Visualize the first 3 results\nfor idx in range(3):\n    if idx >= len(final_predictions): break\n    \n    coords = final_predictions[idx]\n    bg_image = target_images[idx] # Or load the corresponding micrograph if available\n    \n    plt.figure(figsize=(10, 10))\n    plt.imshow(bg_image, cmap=\'gray\')\n    \n    if coords:\n        coords_np = np.array(coords)\n        # coords are [x, y, score]\n        plt.scatter(coords_np[:, 0], coords_np[:, 1], c=\'red\', s=20, marker=\'x\', label=\'Prediction\')\n        \n    plt.title(f"Test Image {idx} | Count: {len(coords)}")\n    plt.axis(\'off\')\n    plt.legend()\n    plt.show()\n    '

In [ ]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in nms_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [ ]:
# ==========================================
# 3. SAVE RESULTS TO .STAR FILE
# ==========================================

# Clean filenames (remove .npy extension to get the base name)
# Assuming 'val_filenames' contains ['image_001.npy', ...]

# BORDER: Ensure this matches your cropping logic (if any).
# If you didn't crop, set to 0. If you cropped 20px, set to 20.
if 'BORDER' not in locals():
    BORDER = 0
    print("⚠️ BORDER variable not found. Defaulting to 0.")

df = pd.DataFrame()

# Iterate through filenames and corresponding predictions
for idx, fname in enumerate(fnames):

    # Get the list of particles for this image [x, y, score]
    particles = final_predictions[idx]

    if not particles:
        continue

    # Create adjusted list of tuples (x + BORDER, y + BORDER)
    adjusted_c_list = [(p[0] + BORDER, p[1] + BORDER) for p in particles]

    # Create temporary DataFrame
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])

    temp_df['rlnImageName'] = [f"{i:06d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # Append to main DataFrame
    if df.empty:
        df = temp_df
    else:
        df = pd.concat([df, temp_df], ignore_index=True)

# Write to .star file
save_path = os.path.join(RESULT_DIR, 'watershed_particles.star')
starfile.write(df, save_path)

print(f"✅ Saved .star file to: {save_path}")
print(f"   Total Particles: {len(df)}")
print(f"   Micrographs Processed: {len(fnames)}")
print(df.head())

✅ Saved .star file to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_test/10017/unet_eb5_dice_CRF/watershed_particles.star
   Total Particles: 46316
   Micrographs Processed: 84
   rlnCoordinateX  rlnCoordinateY                             rlnImageName  \
0            3356             779  000000@Falcon_2012_06_12-15_43_48_0.mrc   
1             549            1206  000001@Falcon_2012_06_12-15_43_48_0.mrc   
2            1771             418  000002@Falcon_2012_06_12-15_43_48_0.mrc   
3             388            1836  000003@Falcon_2012_06_12-15_43_48_0.mrc   
4            2251             569  000004@Falcon_2012_06_12-15_43_48_0.mrc   

                  rlnMicrographName  rlnAutopickFigureOfMerit  
0  Falcon_2012_06_12-15_43_48_0.mrc                  0.996078  
1  Falcon_2012_06_12-15_43_48_0.mrc                  0.996078  
2  Falcon_2012_06_12-15_43_48_0.mrc                  0.996078  
3  Falcon_2012_06_12-15_43_48_

In [ ]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName,rlnAutopickFigureOfMerit
0,3356,779,000000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc,0.996078
1,549,1206,000001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc,0.996078
2,1771,418,000002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc,0.996078
3,388,1836,000003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc,0.996078
4,2251,569,000004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc,0.996078
...,...,...,...,...,...
46311,1820,351,000513@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc,0.996078
46312,621,1070,000514@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc,0.537255
46313,506,3546,000515@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc,0.996078
46314,556,2133,000516@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc,0.996078


In [ ]:
# @markdown ---
# @markdown time used
end_time = time.time()
print(f"End time recorded: {end_time}")

elapsed_time = end_time - start_time
elapsed_time = elapsed_time


hours = int(elapsed_time // 3600)
remaining_seconds = elapsed_time % 3600

minutes = int(remaining_seconds // 60)
seconds = round(remaining_seconds % 60, 3)

print(f"Time spend : {hours} h, {minutes} m, {seconds} s")


gpu_used = "L4" # @param ["CPU high", "T4", "T4 high", "L4"]
per_unit_cost_dict = {"L4" : 1.71, "T4 high" : 1.41, "T4" : 1.19, "CPU high" :  0.24}
per_unit_cost = per_unit_cost_dict[gpu_used]
print(f"unit price per hr {per_unit_cost}")

cost_units = per_unit_cost * elapsed_time / 3600

per_unit_US = 10.49 / 100

cost_price_US = cost_units * per_unit_US

print(f"unit cost : {round(cost_units, 4)}")
print(f"unit price US: {cost_price_US}")
print(f"unit price NTD: {cost_price_US * 30.76}")

End time recorded: 1765869782.0918818
Time spend : 0 h, 32 m, 54.825 s
unit price per hr 1.71
unit cost : 0.938
unit price US: 0.0984005950183171
unit price NTD: 3.0268023027634343
